## Compositional Decomposition Analysis

Loads precomputed results from `scripts/analyze_composition.py` and produces:
1. Residual vs number-of-features curve per label (headline compositionality figure)
2. Label subspace alignment heatmap across labels
3. Feature → content mapping table per label

In [ ]:
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

matplotlib.use("module://matplotlib_inline.backend_inline")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

-- Config --

In [ ]:
MODEL        = "test_01"
EXPERIMENTS  = Path("experiments")
MODEL_DIR    = EXPERIMENTS / MODEL
RESULTS_DIR  = MODEL_DIR / "results"
ANALYSIS_DIR = MODEL_DIR / "analysis"
FIGURES_DIR  = RESULTS_DIR / "composition" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

-- Load Results --

In [ ]:
with open(RESULTS_DIR / "composition.json") as f:
    results = json.load(f)

npz = dict(np.load(RESULTS_DIR / "composition.npz", allow_pickle=True))

alignment_matrix = npz["alignment_matrix"]
label_names      = npz["label_names"].tolist()

print(f"Labels: {label_names}")
print(f"Alignment matrix: {alignment_matrix.shape}")

### 1. Residual vs Number of Features

At each step, the SAE feature whose decoder direction maximally increases
principal-angle coverage of the label subspace is added. The residual
(1 - coverage) shows how quickly each label's subspace can be reconstructed
from SAE decoder directions. Fewer features = more monosemantic.

In [ ]:
from src.analysis.plotting import plot_residual_curves, show_or_savefig

decomposition = results.get("decomposition", {})
if decomposition:
    plot_residual_curves(
        decomposition,
        show=True,
        save_path=FIGURES_DIR / "residual_curves",
    )
else:
    print("No decomposition results found")

### 2. Label Subspace Alignment Heatmap

Entry (i,j) = mean cosine of principal angles between the supervised
subspaces of label i and label j. Values near 1.0 mean the labels
occupy the same directions in z_enc; near 0 means orthogonal concepts.

In [ ]:
from src.analysis.plotting import plot_label_alignment_heatmap

plot_label_alignment_heatmap(
    alignment_matrix, label_names,
    show=True,
    save_path=FIGURES_DIR / "label_alignment_heatmap",
)

### 3. Feature → Content Mapping Table

For each label, lists the SAE features selected by greedy matching pursuit
(in order) and their clinical content labels from `inspect_sae_feature_content`.

Load feature cards if available (from analyze_features.py output)

In [ ]:
from src.analysis.plotting import plot_feature_label_table

# Load feature cards if available (from analyze_features.py output)
feature_cards = None
features_json = ANALYSIS_DIR / "features.json"
if features_json.exists():
    with open(features_json) as f:
        feat_results = json.load(f)
    # Feature cards may be stored separately; if not, pass None
    feature_cards_path = ANALYSIS_DIR / "feature_cards.json"
    if feature_cards_path.exists():
        with open(feature_cards_path) as f:
            feature_cards = json.load(f)

plot_feature_label_table(
    decomposition,
    feature_cards=feature_cards,
    show=True,
    save_path=FIGURES_DIR / "feature_label_table",
)